
# Tablas de distribuciones acumuladas — Paper 3

Este notebook reconstruye las tablas de distribuciones acumuladas que aparecen en `Paper3.tex` y agrega la clasificación de dominancia estocástica.

**Convención utilizada**
- \(D_1\): el modelo secuencial domina al estático, es decir, \(F_{\rm seq}(k)\leq F_{\rm stat}(k)\) para todo \(k\).
- \(D_2\): el modelo estático domina al secuencial, es decir, \(F_{\rm stat}(k)\leq F_{\rm seq}(k)\) para todo \(k\).
- **Ambas**: las dos funciones de distribución coinciden.
- **Ninguna**: las funciones de distribución se cruzan.

Los valores de las tablas se expresan como fracciones exactas (`Fraction`), no como aproximaciones decimales.


In [ ]:
from fractions import Fraction
import pandas as pd
from IPython.display import display, Markdown

def parse_fraction(value):
    """Convierte enteros o cadenas como r'\\frac{1}{3}' en Fraction."""
    if isinstance(value, Fraction):
        return value
    value = str(value).strip()
    if value == "1":
        return Fraction(1, 1)
    if value == "0":
        return Fraction(0, 1)
    if value.startswith(r"\frac"):
        nums = [int(x) for x in __import__("re").findall(r"\d+", value)]
        return Fraction(nums[0], nums[1])
    return Fraction(value)

def classify_cdfs(seq_cdf, stat_cdf):
    """Clasifica la dominancia comparando ambas CDF en todos los puntos de salto."""
    ks = sorted(set(seq_cdf) | set(stat_cdf))
    # Fuera del soporte, la CDF se mantiene constante. Para cada k se toma
    # el último valor acumulado disponible hasta k.
    def value_at(cdf, k):
        eligible = [j for j in cdf if j <= k]
        return cdf[max(eligible)] if eligible else Fraction(0, 1)

    seq_vals = [value_at(seq_cdf, k) for k in ks]
    stat_vals = [value_at(stat_cdf, k) for k in ks]
    d1 = all(s <= t for s, t in zip(seq_vals, stat_vals))
    d2 = all(t <= s for s, t in zip(seq_vals, stat_vals))
    if d1 and d2:
        return "Ambas"
    if d1:
        return "D1: secuencial domina"
    if d2:
        return "D2: estático domina"
    return "Ninguna"

def cdf_from_dict(data):
    return {int(k): parse_fraction(v) for k, v in data.items()}

def format_cdf(cdf):
    return "; ".join(
        f"k={k}: {v.numerator}/{v.denominator}" if v.denominator != 1
        else f"k={k}: {v.numerator}"
        for k, v in sorted(cdf.items())
    )

def table_from_cdf_rows(rows, graph_col=None):
    records = []
    for row in rows:
        seq = cdf_from_dict(row["secuencial"])
        stat = cdf_from_dict(row["estatico"])
        record = {
            "Grafo": row.get("grafo", ""),
            "Valores de k": ", ".join(map(str, sorted(set(seq) | set(stat)))),
            "CDF secuencial": format_cdf(seq),
            "CDF estática": format_cdf(stat),
            "Dominancia": classify_cdfs(seq, stat),
        }
        records.append(record)
    df = pd.DataFrame(records)
    if graph_col is False:
        df = df.drop(columns=["Grafo"])
    return df


## 1. Grafos conexos con menos de 6 vértices

La tabla del artículo excluye grafos completos, caminos y ciclos. Se transcriben los 21 grafos que aparecen en la tabla del `.tex`. La columna `Grafo` conserva el nombre de archivo de la imagen original; las imágenes no están incluidas en los archivos proporcionados, por lo que aquí se muestra su identificador.

In [ ]:
small_graphs = [{'grafo': 'grafo_05', 'secuencial': {'1': '\\frac{1}{4}', '3': '1'}, 'estatico': {'1': '\\frac{1}{2}', '3': '1'}}, {'grafo': 'grafo_07', 'secuencial': {'1': '\\frac{1}{4}', '2': '1'}, 'estatico': {'1': '\\frac{1}{3}', '2': '1'}}, {'grafo': 'grafo_09', 'secuencial': {'1': '\\frac{1}{2}', '2': '1'}, 'estatico': {'1': '\\frac{2}{3}', '2': '1'}}, {'grafo': 'grafo_11', 'secuencial': {'1': '\\frac{1}{5}', '4': '1'}, 'estatico': {'1': '\\frac{1}{2}', '4': '1'}}, {'grafo': 'grafo_12', 'secuencial': {'2': '\\frac{4}{15}', '3': '1'}, 'estatico': {'2': '\\frac{1}{3}', '3': '1'}}, {'grafo': 'grafo_14', 'secuencial': {'1': '\\frac{1}{5}', '3': '1'}, 'estatico': {'1': '\\frac{1}{3}', '3': '1'}}, {'grafo': 'grafo_15', 'secuencial': {'2': '\\frac{8}{15}', '3': '1'}, 'estatico': {'2': '\\frac{2}{3}', '3': '1'}}, {'grafo': 'grafo_16', 'secuencial': {'2': '1'}, 'estatico': {'2': '1'}}, {'grafo': 'grafo_17', 'secuencial': {'2': '\\frac{7}{15}', '3': '1'}, 'estatico': {'2': '\\frac{2}{3}', '3': '1'}}, {'grafo': 'grafo_19', 'secuencial': {'1': '\\frac{1}{5}', '2': '\\frac{7}{15}', '3': '1'}, 'estatico': {'1': '\\frac{1}{3}', '2': '\\frac{2}{3}', '3': '1'}}, {'grafo': 'grafo_20', 'secuencial': {'2': '1'}, 'estatico': {'2': '1'}}, {'grafo': 'grafo_21', 'secuencial': {'1': '\\frac{1}{5}', '2': '1'}, 'estatico': {'1': '\\frac{1}{5}', '2': '1'}}, {'grafo': 'grafo_22', 'secuencial': {'2': '1'}, 'estatico': {'2': '1'}}, {'grafo': 'grafo_23', 'secuencial': {'2': '\\frac{2}{5}', '3': '1'}, 'estatico': {'2': '\\frac{1}{2}', '3': '1'}}, {'grafo': 'grafo_24', 'secuencial': {'1': '\\frac{1}{5}', '2': '1'}, 'estatico': {'1': '\\frac{1}{4}', '2': '1'}}, {'grafo': 'grafo_25', 'secuencial': {'1': '\\frac{2}{5}', '3': '1'}, 'estatico': {'1': '\\frac{2}{3}', '3': '1'}}, {'grafo': 'grafo_26', 'secuencial': {'1': '\\frac{1}{5}', '2': '1'}, 'estatico': {'1': '\\frac{1}{4}', '2': '1'}}, {'grafo': 'grafo_27', 'secuencial': {'2': '1'}, 'estatico': {'2': '1'}}, {'grafo': 'grafo_28', 'secuencial': {'1': '\\frac{2}{5}', '2': '1'}, 'estatico': {'1': '\\frac{1}{2}', '2': '1'}}, {'grafo': 'grafo_29', 'secuencial': {'1': '\\frac{1}{5}', '2': '1'}, 'estatico': {'1': '\\frac{1}{3}', '2': '1'}}, {'grafo': 'grafo_30', 'secuencial': {'1': '\\frac{3}{5}', '2': '1'}, 'estatico': {'1': '\\frac{3}{4}', '2': '1'}}]

tabla_grafos_pequenos = table_from_cdf_rows(small_graphs)
display(tabla_grafos_pequenos)

print('Clasificación global:')
display(tabla_grafos_pequenos[['Grafo', 'Dominancia']])

## 2. Contraejemplo conexo de 8 vértices

Esta es la tabla asociada a la Figura `standalone_05` del artículo. Los valores se obtienen de las distribuciones exactas:

In [ ]:
grafo_8_vertices = {
    "grafo": "Contraejemplo conexo de 8 vértices",
    "secuencial": {"2": r"\frac{3}{20}", "3": r"\frac{5}{12}", "4": "1"},
    "estatico": {"2": r"\frac{1}{9}", "3": r"\frac{5}{9}", "4": "1"},
}

tabla_grafo_8 = table_from_cdf_rows([grafo_8_vertices])
display(tabla_grafo_8)

## 3. Ejemplo 1: grafo desconectado

En este ejemplo el modelo estático domina al secuencial (`D2`).

In [ ]:
ejemplo_1 = {
    "grafo": "Ejemplo 1",
    "secuencial": {"4": r"\frac{8}{105}", "5": r"\frac{50}{105}", "6": "1"},
    "estatico": {"4": r"\frac{1}{15}", "5": r"\frac{7}{15}", "6": "1"},
}
tabla_ejemplo_1 = table_from_cdf_rows([ejemplo_1])
display(tabla_ejemplo_1)

## 4. Ejemplo 2: grafo desconectado

En este ejemplo el modelo secuencial domina al estático (`D1`).

In [ ]:
ejemplo_2 = {
    "grafo": "Ejemplo 2",
    "secuencial": {"3": r"\frac{5}{72}", "4": r"\frac{17}{36}", "5": "1"},
    "estatico": {"3": r"\frac{1}{10}", "4": r"\frac{3}{5}", "5": "1"},
}
tabla_ejemplo_2 = table_from_cdf_rows([ejemplo_2])
display(tabla_ejemplo_2)

## 5. Ejemplo 3: grafo desconectado

Las funciones de distribución se cruzan, por lo que no hay dominancia estocástica.

In [ ]:
ejemplo_3 = {
    "grafo": "Ejemplo 3",
    "secuencial": {"3": r"\frac{2}{35}", "4": r"\frac{47}{105}", "5": "1"},
    "estatico": {"3": r"\frac{1}{18}", "4": r"\frac{5}{9}", "5": "1"},
}
tabla_ejemplo_3 = table_from_cdf_rows([ejemplo_3])
display(tabla_ejemplo_3)

## 6. Resumen de todas las clasificaciones

Esta última tabla reúne el resultado de dominancia de cada caso presentado en las tablas anteriores.

In [ ]:
resumen = pd.concat([
    tabla_grafos_pequenos[["Grafo", "Dominancia"]],
    tabla_grafo_8[["Grafo", "Dominancia"]],
    tabla_ejemplo_1[["Grafo", "Dominancia"]],
    tabla_ejemplo_2[["Grafo", "Dominancia"]],
    tabla_ejemplo_3[["Grafo", "Dominancia"]],
], ignore_index=True)

display(resumen)
